# M01B · Sequence Manipulation: Composition, Transcription, Translation & Motifs (Solutions)
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M01_Python_Basics/solutions/M01B_Sequence_Manipulation_Solutions.ipynb)

⏱ ~120 min · Difficulty: ★★☆ · Prerequisites: M01A_Core_Python


## 1. Learning Outcomes

By the end of this notebook, you will be able to:
1. **Validate** DNA and RNA sequences against canonical and degenerate IUPAC nucleotide alphabets.
2. **Calculate** nucleotide frequencies, GC content, and strand-specific AT/GC skews to characterize replication origins and genomic composition.
3. **Construct** reverse complement strands using dictionary mappings and high-performance C-level `str.maketrans()` translations.
4. **Simulate** transcription (DNA to RNA) and reading frame-specific translation using the Standard Genetic Code dictionary.
5. **Implement** biophysical calculations including single-stranded molecular weight and primer melting temperatures ($T_m$) using the Wallace rule and Marmur-Doty equation.
6. **Extract and quantify** sequence motifs and k-mer frequency spectra, distinguishing overlapping from non-overlapping occurrences.
7. **Filter** high-throughput sequence libraries using multi-parameter quality thresholds and evaluate primer-dimer interactions.


## 2. Prerequisites

- Completion of **M01A_Core_Python** (variables, strings, lists, dicts, loops, and functions).
- Understanding of the Central Dogma of Molecular Biology: $\text{DNA} \to \text{mRNA} \to \text{Protein}$.


## 3. Setup

Run this cell to detect the runtime environment, configure paths, and import standard modules.


In [ ]:
# --- Setup -------------------------------------------------------
import sys
import os
import platform
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Cloning repository...")
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    p = Path.cwd()
    found_root = None
    for candidate in [p] + list(p.parents):
        if (candidate / "data" / "sequences").exists() or (candidate / "COURSE_PLAN.md").exists():
            found_root = candidate
            break
    if found_root:
        ROOT = found_root
    else:
        repo_candidate = Path("/Users/prateekpaul/Library/CloudStorage/GoogleDrive-prateekp@iiitd.ac.in/My Drive/IIITD/mygithub/ProgrammingCB")
        ROOT = repo_candidate if repo_candidate.exists() else p

DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / "M01B"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Python version     : {sys.version.split()[0]} ({platform.python_implementation()})")
print(f"✓ Repository Root    : {ROOT.resolve()}")
print(f"✓ Data Directory     : {DATA_DIR.resolve()}")
print(f"✓ Output Directory   : {OUT_DIR.resolve()}")


## 4. Concept Sections

---

### 1B.1 Sequence Validation

#### 4.1.1 Theory: Quality Assurance for Sequence Data
Before running downstream analytical pipelines, sequences must be validated. Raw sequencing data often contains lowercase characters, uncalled bases ('N'), or degenerate IUPAC codes.
A strict validator ensures every base belongs to $\{A, C, G, T\}$, while a lenient validator permits degenerate IUPAC characters.


In [ ]:
# Worked Example: Robust DNA Validator
def is_valid_dna(seq: str, allow_ambiguous: bool = False) -> bool:
    """Validates whether a string consists of legal DNA nucleotide characters."""
    if not seq:
        return False
    clean_seq = seq.upper().strip()
    if allow_ambiguous:
        valid_alphabet = set("ACGTRYSWKMBDHVN")
    else:
        valid_alphabet = set("ACGT")
    return all(base in valid_alphabet for base in clean_seq)

print("is_valid_dna('ATGCGT')        :", is_valid_dna("ATGCGT"))
print("is_valid_dna('ATGCNGT', True) :", is_valid_dna("ATGCNGT", allow_ambiguous=True))
assert is_valid_dna("ATGCGT") is True
assert is_valid_dna("ATGCNGT", allow_ambiguous=False) is False


---

### 1B.2 Sequence Composition, GC Content & GC Skew

#### 4.2.1 Theory: Genomic Thermostability and Replication Asymmetry
- **GC Content**: Percentage of bases that are $G$ or $C$: $\text{GC}\% = \frac{G + C}{A + T + G + C} \times 100$.
- **GC Skew**: $(G - C) / (G + C)$. Asymmetric nucleotide substitution between leading and lagging strands of DNA replication indicates the bacterial **Origin of Replication (*oriC*)**!


In [ ]:
# Worked Example: GC Content and GC Skew Calculations
def calculate_gc_content(seq: str) -> float:
    """Calculates GC percentage of a DNA sequence."""
    clean = seq.upper()
    total_bases = clean.count("A") + clean.count("C") + clean.count("G") + clean.count("T")
    if total_bases == 0:
        return 0.0
    gc = clean.count("G") + clean.count("C")
    return (gc / total_bases) * 100.0

def calculate_gc_skew(seq: str) -> float:
    """Calculates GC Skew: (G - C) / (G + C)."""
    clean = seq.upper()
    g = clean.count("G")
    c = clean.count("C")
    if (g + c) == 0:
        return 0.0
    return (g - c) / (g + c)

sample = "GGGCCCGGG"
print(f"Sequence: {sample} | GC%: {calculate_gc_content(sample):.1f}% | Skew: {calculate_gc_skew(sample):.4f}")
assert calculate_gc_content("GCGC") == 100.0
assert calculate_gc_skew("GGGCC") == round((3 - 2) / 5, 4)


---

### 1B.3 Complement & Reverse Complement

#### 4.3.1 Theory: The Antiparallel Double Helix
DNA strands are directional ($5' \to 3'$) and antiparallel. To analyze the opposite strand, we complement each base and reverse the sequence string ($[::-1]$).


In [ ]:
# High-Speed Reverse Complement with str.maketrans
_COMP_TRANS = str.maketrans("ACGTUacgtuNn", "TGCAAtgcaaNn")

def reverse_complement(seq: str) -> str:
    """Generates 5'->3' reverse complement."""
    return seq.translate(_COMP_TRANS)[::-1]

test_seq = "ATGCGT"
rc = reverse_complement(test_seq)
print(f"5'-{test_seq}-3' -> 5'-{rc}-3'")
assert rc == "ACGCAT"


---

### 1B.4 Transcription: DNA to RNA

#### 4.4.1 Theory: Coding vs. Template Strands
Coding strand ($5' \to 3'$) to mRNA: replace Thymine (`T`) with Uracil (`U`).


In [ ]:
def transcribe(dna_seq: str) -> str:
    """Transcribes DNA coding strand to mRNA."""
    return dna_seq.upper().replace("T", "U")

def reverse_transcribe(rna_seq: str) -> str:
    """Converts mRNA back to cDNA (U -> T)."""
    return rna_seq.upper().replace("U", "T")

print("Coding DNA -> mRNA:", transcribe("ATGCGTACTGAC"))
assert transcribe("ATGC") == "AUGC"
assert reverse_transcribe("AUGC") == "ATGC"


---

### 1B.5 Translation Across Reading Frames

#### 4.5.1 Theory: The Universal Genetic Code
Maps 64 triplet codons to 20 amino acids and 3 stops.


In [ ]:
STANDARD_GENETIC_CODE = {
    'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
    'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
    'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
    'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
    'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
    'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
    'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
    'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
    'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
    'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
    'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
    'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
    'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
    'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
    'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*',
    'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W',
}

def translate_dna(dna_seq: str, frame: int = 0, to_stop: bool = False) -> str:
    clean = dna_seq.upper().replace("U", "T")
    peptide = []
    for i in range(frame, len(clean) - 2, 3):
        codon = clean[i:i+3]
        aa = STANDARD_GENETIC_CODE.get(codon, "X")
        if to_stop and aa == "*":
            break
        peptide.append(aa)
    return "".join(peptide)

print("Translation:", translate_dna("ATGGCCATTGTAATGGGCCGCTGA", to_stop=True))
assert translate_dna("ATGGCCATTGTAATGGGCCGCTGA", to_stop=True) == "MAIVMGR"


---

### 1B.6 String-Based Biophysical Calculations

#### 4.6.1 Theory: Melting Temperature & Molecular Weight
- **Wallace Rule** ($< 14$ nt): $T_m = 2(A + T) + 4(G + C)$.
- **Marmur-Doty** ($\ge 14$ nt): $T_m = 64.9 + 41.0 \times \frac{(G + C) - 16.4}{N}$.
- **ssDNA MW**: $(A \times 313.21 + T \times 304.20 + C \times 289.18 + G \times 329.21) - 61.96$ g/mol.


In [ ]:
def calculate_tm(seq: str) -> float:
    clean = seq.upper()
    n = len(clean)
    if n == 0: return 0.0
    at = clean.count("A") + clean.count("T")
    gc = clean.count("G") + clean.count("C")
    if n < 14:
        return float(2 * at + 4 * gc)
    return round(64.9 + 41.0 * (gc - 16.4) / n, 2)

print("Tm ('ATGC'):", calculate_tm("ATGC"))
assert calculate_tm("ATGC") == 12.0


---

### 1B.7 Sequence Motifs & K-mers

#### 4.7.1 Theory: Substring Profiling in Genomics
Overlapping vs non-overlapping motif search; k-mer frequency distribution.


In [ ]:
from collections import Counter

def kmer_frequencies(seq: str, k: int) -> dict[str, int]:
    clean = seq.upper()
    kmers = [clean[i:i+k] for i in range(len(clean) - k + 1)]
    return dict(Counter(kmers))

def find_motif(seq: str, motif: str, overlapping: bool = True) -> list[int]:
    s = seq.upper()
    m = motif.upper()
    pos, start = [], 0
    while True:
        idx = s.find(m, start)
        if idx == -1: break
        pos.append(idx)
        start = idx + 1 if overlapping else idx + len(m)
    return pos

print("K-mers:", kmer_frequencies("ATATATAGC", 3))
print("Motifs:", find_motif("ATATATAGC", "ATA", overlapping=True))
assert find_motif("ATATATAGC", "ATA", overlapping=True) == [0, 2, 4]


---

### 1B.8 Filtering Sequences

#### 4.8.1 Theory: Quality Trimming in High-Throughput Pipelines
Multi-parameter QC filtering on length, GC% bounds, and maximum ambiguous bases.


In [ ]:
def filter_sequences(seq_list: list[str], min_len: int = 15, 
                     min_gc: float = 40.0, max_gc: float = 65.0, 
                     max_n: int = 0) -> list[str]:
    filtered = []
    for s in seq_list:
        clean = s.upper()
        if len(clean) < min_len: continue
        if clean.count("N") > max_n: continue
        gc = calculate_gc_content(clean)
        if min_gc <= gc <= max_gc: filtered.append(clean)
    return filtered

passed = filter_sequences(["ATGCGTACGTTTGCAACCGT", "ATGC"])
assert len(passed) == 1


## 5. Practice Exercises

Solve each of the 15 exercises below.


### Exercise 1 (Level 1 ★): Strict DNA Sequence Validator

**Problem**: Write a function `validate_dna_sequence(seq: str, allow_n: bool = False) -> bool` verifying characters belong to ACGT (or ACGTN if allow_n=True).

- **Input**: String `seq`, boolean `allow_n`.
- **Expected Output**: Boolean `True` or `False`.


In [ ]:
def validate_dna_sequence(seq: str, allow_n: bool = False) -> bool:
    if not seq: return False
    clean = seq.upper().strip()
    valid = set("ACGTN" if allow_n else "ACGT")
    return all(b in valid for b in clean)


#### 💡 Solution Explanation
Guards against empty strings and checks characters against the allowed set.

In [ ]:
assert validate_dna_sequence('ATGC') is True
assert validate_dna_sequence('ATGCN', False) is False
print('✓ Exercise 1 passed!')


### Exercise 2 (Level 1 ★): GC Content Percentage Calculator

**Problem**: Write a function `calculate_gc_percentage(seq: str) -> float` computing GC% rounded to 2 decimals.

- **Input**: String `seq`.
- **Expected Output**: Float GC% (0.0 to 100.0).


In [ ]:
def calculate_gc_percentage(seq: str) -> float:
    clean = seq.upper()
    tot = clean.count('A') + clean.count('C') + clean.count('G') + clean.count('T')
    if tot == 0: return 0.0
    return round(((clean.count('G') + clean.count('C')) / tot) * 100.0, 2)


#### 💡 Solution Explanation
Normalizes G+C count by canonical nucleotide sum and rounds to 2 decimals.

In [ ]:
assert calculate_gc_percentage('ATGC') == 50.0
assert calculate_gc_percentage('GCGC') == 100.0
print('✓ Exercise 2 passed!')


### Exercise 3 (Level 1 ★): Reverse Complement Using str.maketrans

**Problem**: Write a function `get_reverse_complement(seq: str) -> str` generating 5'->3' reverse complement.

- **Input**: String `seq`.
- **Expected Output**: Uppercase reverse complement string.


In [ ]:
def get_reverse_complement(seq: str) -> str:
    trans = str.maketrans("ACGTNacgtn", "TGCANtgcan")
    return seq.translate(trans)[::-1].upper()


#### 💡 Solution Explanation
C-speed character translation and slice reversal.

In [ ]:
assert get_reverse_complement('ATGC') == 'GCAT'
assert get_reverse_complement('ATGCN') == 'NGCAT'
print('✓ Exercise 3 passed!')


### Exercise 4 (Level 1 ★): DNA Coding Strand to mRNA Transcription

**Problem**: Write a function `transcribe_dna_to_rna(dna_seq: str) -> str` converting DNA coding strand to mRNA.

- **Input**: DNA string `dna_seq`.
- **Expected Output**: mRNA string.


In [ ]:
def transcribe_dna_to_rna(dna_seq: str) -> str:
    return dna_seq.upper().replace('T', 'U')


#### 💡 Solution Explanation
Substitutes T with U in the coding sequence.

In [ ]:
assert transcribe_dna_to_rna('ATGCGT') == 'AUGCGU'
print('✓ Exercise 4 passed!')


### Exercise 5 (Level 2 ★★): Melting Temperature Dispatcher

**Problem**: Write a function `calculate_melting_temp(seq: str) -> float` using Wallace (<14 bp) or Marmur-Doty (>=14 bp).

- **Input**: String `seq`.
- **Expected Output**: Float Tm in °C.


In [ ]:
def calculate_melting_temp(seq: str) -> float:
    clean = seq.upper()
    n = len(clean)
    if n == 0: return 0.0
    at = clean.count('A') + clean.count('T')
    gc = clean.count('G') + clean.count('C')
    if n < 14: return float(2 * at + 4 * gc)
    return round(64.9 + 41.0 * (gc - 16.4) / n, 2)


#### 💡 Solution Explanation
Length-based formula dispatch.

In [ ]:
assert calculate_melting_temp('ATGC') == 12.0
assert calculate_melting_temp('ATGCATGCATGCAT') == round(64.9 + 41.0 * (6 - 16.4) / 14, 2)
print('✓ Exercise 5 passed!')


### Exercise 6 (Level 2 ★★): Three-Frame Translation

**Problem**: Write a function `translate_three_frames(seq: str, to_stop: bool = False) -> dict[int, str]` translating across frames 0, 1, 2.

- **Input**: DNA string `seq`.
- **Expected Output**: Dictionary `{0: str, 1: str, 2: str}`.


In [ ]:
def translate_three_frames(seq: str, to_stop: bool = False) -> dict[int, str]:
    return {f: translate_dna(seq, frame=f, to_stop=to_stop) for f in (0, 1, 2)}


#### 💡 Solution Explanation
Calls `translate_dna()` across frames 0, 1, and 2.

In [ ]:
res = translate_three_frames('ATGGCCATTGTA')
assert res[0] == 'MAIV'
print('✓ Exercise 6 passed!')


### Exercise 7 (Level 2 ★★): Top K-mers Detector

**Problem**: Write a function `top_kmers(seq: str, k: int, top_n: int = 3) -> list[tuple[str, int]]` returning top_n k-mers sorted by count desc, kmer asc.

- **Input**: String `seq`, integers `k`, `top_n`.
- **Expected Output**: List of `(kmer, count)` tuples.


In [ ]:
def top_kmers(seq: str, k: int, top_n: int = 3) -> list[tuple[str, int]]:
    clean = seq.upper()
    if len(clean) < k or k <= 0: return []
    counts = {}
    for i in range(len(clean) - k + 1):
        km = clean[i:i+k]
        counts[km] = counts.get(km, 0) + 1
    sorted_km = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
    return sorted_km[:top_n]


#### 💡 Solution Explanation
Sliding window frequency count with composite sorting tuple `(-count, kmer)`.

In [ ]:
top2 = top_kmers('ATATATGCGC', 2, 2)
assert top2[0] == ('AT', 3)
print('✓ Exercise 7 passed!')


### Exercise 8 (Level 2 ★★): AT and GC Skew Profile

**Problem**: Write a function `calculate_skews(seq: str) -> dict[str, float]` returning `{'gc_skew': float, 'at_skew': float}` rounded to 4 decimals.

- **Input**: String `seq`.
- **Expected Output**: Dictionary with skews.


In [ ]:
def calculate_skews(seq: str) -> dict[str, float]:
    clean = seq.upper()
    g, c, a, t = clean.count('G'), clean.count('C'), clean.count('A'), clean.count('T')
    gc_s = round((g - c)/(g + c), 4) if (g + c) > 0 else 0.0
    at_s = round((a - t)/(a + t), 4) if (a + t) > 0 else 0.0
    return {'gc_skew': gc_s, 'at_skew': at_s}


#### 💡 Solution Explanation
Calculates $(G-C)/(G+C)$ and $(A-T)/(A+T)$ with zero-division protection.

In [ ]:
res = calculate_skews('GGGCCC')
assert res['gc_skew'] == 0.0
print('✓ Exercise 8 passed!')


### Exercise 9 (Level 3 ★★★): Molecular Weight Calculator

**Problem**: Write a function `compute_molecular_weight(seq: str, molecule: str = 'dna') -> float` computing ssDNA or peptide MW in g/mol rounded to 2 decimals.

- **Input**: String `seq`, string `molecule` ('dna'/'protein').
- **Expected Output**: Float MW.


In [ ]:
def compute_molecular_weight(seq: str, molecule: str = 'dna') -> float:
    clean = seq.upper()
    if not clean: return 0.0
    if molecule.lower() == 'dna':
        mw = (clean.count('A')*313.21 + clean.count('T')*304.20 + clean.count('C')*289.18 + clean.count('G')*329.21) - 61.96
        return round(mw, 2)
    elif molecule.lower() == 'protein':
        aa_w = {'A':89.09,'R':174.20,'N':132.12,'D':133.10,'C':121.15,'E':147.13,'Q':146.15,'G':75.07,'H':155.16,'I':131.17,'L':131.17,'K':146.19,'M':149.21,'F':165.19,'P':115.13,'S':105.09,'T':119.12,'W':204.23,'Y':181.19,'V':117.15}
        tot = sum(aa_w.get(aa, 0.0) for aa in clean)
        loss = (len(clean) - 1) * 18.015
        return round(tot - loss, 2)
    raise ValueError("Invalid molecule type")


#### 💡 Solution Explanation
Biochemical mass sum for ssDNA (minus dehydration 61.96) or peptide (minus $(N-1) \times 18.015$).

In [ ]:
assert compute_molecular_weight('ATGC', 'dna') == 1173.84
assert abs(compute_molecular_weight('AG', 'protein') - 146.14) <= 0.01
print('✓ Exercise 9 passed!')


### Exercise 10 (Level 3 ★★★): NGS Read Filter with Audit Report

**Problem**: Write a function `filter_ngs_library(reads: list[str], min_len: int = 50, max_len: int = 150, min_gc: float = 40.0, max_gc: float = 60.0, max_n: int = 0) -> dict` returning audit report dictionary.

- **Input**: List of strings and filter thresholds.
- **Expected Output**: Report dictionary.


In [ ]:
def filter_ngs_library(reads: list[str], min_len: int = 50, max_len: int = 150, min_gc: float = 40.0, max_gc: float = 60.0, max_n: int = 0) -> dict:
    tot = len(reads)
    passed = []
    for r in reads:
        c = r.upper().strip()
        if not (min_len <= len(c) <= max_len): continue
        if c.count('N') > max_n: continue
        gc = calculate_gc_content(c)
        if not (min_gc <= gc <= max_gc): continue
        passed.append(c)
    return {'passed_reads': passed, 'total_input': tot, 'passed_count': len(passed), 'pass_rate': round((len(passed)/tot)*100.0, 2) if tot else 0.0}


#### 💡 Solution Explanation
Applies multi-stage filtering and produces an audit summary dictionary.

In [ ]:
r = filter_ngs_library(['ATGC'*15, 'A'*60], min_len=50, max_len=100, min_gc=40.0, max_gc=60.0)
assert r['passed_count'] == 1
print('✓ Exercise 10 passed!')


### Exercise 11 (Level 1 ★): RNA to cDNA Reverse Transcription

**Problem**: Write a function `rna_to_cdna(rna_seq: str) -> str` converting RNA to cDNA (U -> T, case-insensitive).

- **Input**: RNA string `rna_seq`.
- **Expected Output**: cDNA uppercase string.


In [ ]:
def rna_to_cdna(rna_seq: str) -> str:
    return rna_seq.upper().replace('U', 'T')


#### 💡 Solution Explanation
Direct character substitution replacing Uracil with Thymine.

In [ ]:
assert rna_to_cdna('AUGCGU') == 'ATGCGT'
print('✓ Exercise 11 passed!')


### Exercise 12 (Level 2 ★★): PCR Primer Pair Compatibility Auditor

**Problem**: Write a function `validate_primer_pair(fwd: str, rev: str) -> dict` checking whether a PCR primer pair is compatible: lengths between 18 and 25 bp, GC% between 40.0 and 60.0%, and $|T_{m,fwd} - T_{m,rev}| \le 3.0^\circ\text{C}$. Return `{'is_compatible': bool, 'tm_diff': float, 'fwd_tm': float, 'rev_tm': float}`.

- **Input**: Two primer strings `fwd` and `rev`.
- **Expected Output**: Compatibility dictionary.


In [ ]:
def validate_primer_pair(fwd: str, rev: str) -> dict:
    f, r = fwd.upper(), rev.upper()
    tm_f = calculate_tm(f)
    tm_r = calculate_tm(r)
    diff = round(abs(tm_f - tm_r), 2)
    gc_f = calculate_gc_content(f)
    gc_r = calculate_gc_content(r)
    
    len_ok = (18 <= len(f) <= 25) and (18 <= len(r) <= 25)
    gc_ok = (40.0 <= gc_f <= 60.0) and (40.0 <= gc_r <= 60.0)
    tm_ok = (diff <= 3.0)
    
    return {
        'is_compatible': len_ok and gc_ok and tm_ok,
        'tm_diff': diff,
        'fwd_tm': tm_f,
        'rev_tm': tm_r
    }


#### 💡 Solution Explanation
Audits standard wet-lab PCR primer design constraints: length bounds, GC content, and annealing temperature balance.

In [ ]:
# Pair with length 20, ~50% GC, balanced Tm
f = "ATGGAGGAGCCGCAGTCAGA"
r = "GTCTGAGTCCTGCCCTTCTG"
res = validate_primer_pair(f, r)
assert 'is_compatible' in res
assert res['tm_diff'] <= 3.0
print('✓ Exercise 12 passed!')


### Exercise 13 (Level 2 ★★): Overlapping vs Non-Overlapping Motif Count Comparator

**Problem**: Write a function `compare_motif_counts(seq: str, motif: str) -> dict` returning `{'overlapping': int, 'non_overlapping': int}` counts of `motif` in `seq`.

- **Input**: Strings `seq` and `motif`.
- **Expected Output**: Dictionary with counts.


In [ ]:
def compare_motif_counts(seq: str, motif: str) -> dict:
    s, m = seq.upper(), motif.upper()
    if not m or len(s) < len(m): return {'overlapping': 0, 'non_overlapping': 0}
    ov = len(find_motif(s, m, overlapping=True))
    nov = len(find_motif(s, m, overlapping=False))
    return {'overlapping': ov, 'non_overlapping': nov}


#### 💡 Solution Explanation
Compares search results when incrementing by 1 vs incrementing by `len(motif)`.

In [ ]:
res = compare_motif_counts('AAAA', 'AA')
assert res == {'overlapping': 3, 'non_overlapping': 2}
print('✓ Exercise 13 passed!')


### Exercise 14 (Level 3 ★★★): K-mer Jaccard Similarity Between Sequences

**Problem**: Write a function `kmer_jaccard_similarity(s1: str, s2: str, k: int = 4) -> float` computing the Jaccard similarity between k-mer sets of two sequences: $J(A, B) = \frac{|K_A \cap K_B|}{|K_A \cup K_B|}$. Return float rounded to 4 decimals (or 0.0 if union is empty).

- **Input**: Two DNA strings and integer `k`.
- **Expected Output**: Float Jaccard index (0.0 to 1.0).


In [ ]:
def kmer_jaccard_similarity(s1: str, s2: str, k: int = 4) -> float:
    c1, c2 = s1.upper(), s2.upper()
    if len(c1) < k or len(c2) < k: return 0.0
    k1 = {c1[i:i+k] for i in range(len(c1) - k + 1)}
    k2 = {c2[i:i+k] for i in range(len(c2) - k + 1)}
    union = k1 | k2
    if not union: return 0.0
    return round(len(k1 & k2) / len(union), 4)


#### 💡 Solution Explanation
Alignment-free distance metric used in metagenomics and fast genome clustering (Mash / MinHash).

In [ ]:
assert kmer_jaccard_similarity('ATGCGT', 'ATGCGT', 3) == 1.0
assert kmer_jaccard_similarity('AAAA', 'TTTT', 2) == 0.0
print('✓ Exercise 14 passed!')


### Exercise 15 (Level 3 ★★★): Rolling GC Skew and Bacterial Origin of Replication (oriC) Detector

**Problem**: Write a function `find_oric_ter_candidates(circular_seq: str, window: int = 100, step: int = 50) -> dict` that computes the cumulative GC skew across a circular sequence and returns the 0-based coordinate corresponding to the minimum cumulative skew (candidate *oriC*) and maximum cumulative skew (candidate *ter*). Return `{'oric_candidate_pos': int, 'ter_candidate_pos': int}`.

- **Input**: DNA string `circular_seq`, integers `window`, `step`.
- **Expected Output**: Candidate coordinates dictionary.


In [ ]:
def find_oric_ter_candidates(circular_seq: str, window: int = 100, step: int = 50) -> dict:
    clean = circular_seq.upper()
    n = len(clean)
    if n < window:
        return {'oric_candidate_pos': 0, 'ter_candidate_pos': 0}
        
    cum_skew = 0.0
    skew_profile = []
    
    for i in range(0, n, step):
        # Handle wrap-around for circular genome
        win = clean[i:i+window] if i + window <= n else clean[i:] + clean[:(i+window)%n]
        g, c = win.count('G'), win.count('C')
        local_skew = (g - c) / (g + c) if (g + c) > 0 else 0.0
        cum_skew += local_skew
        skew_profile.append((i, cum_skew))
        
    # Minimum cumulative skew marks oriC; maximum marks ter
    min_pos = min(skew_profile, key=lambda x: x[1])[0]
    max_pos = max(skew_profile, key=lambda x: x[1])[0]
    
    return {'oric_candidate_pos': min_pos, 'ter_candidate_pos': max_pos}


#### 💡 Solution Explanation
Simulates cumulative GC skew along a bacterial replichore, identifying the inflection points corresponding to replication origin and terminus.

In [ ]:
# Circular DNA: first half C-rich (lagging), second half G-rich (leading)
synth = ("C" * 200) + ("G" * 200)
res = find_oric_ter_candidates(synth, window=50, step=25)
assert 'oric_candidate_pos' in res
assert 'ter_candidate_pos' in res
print('✓ Exercise 15 passed!')


## 6. Summary & Cheat Sheet Table

| Operation | Mathematical / Biological Definition | Python Implementation |
|---|---|---|
| **Validation** | Base $\in \{A, C, G, T\}$ | `all(b in set("ACGT") for b in seq.upper())` |
| **GC Content** | $\frac{G + C}{\text{Total}} \times 100$ | `(seq.count('G') + seq.count('C')) / len(seq) * 100` |
| **GC Skew** | $\frac{G - C}{G + C}$ | `(g - c) / (g + c) if (g + c) > 0 else 0.0` |
| **Rev Comp** | $5' \to 3'$ opposite strand | `seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]` |
| **Transcription** | Coding DNA $\to$ mRNA | `dna_seq.replace("T", "U")` |
| **Translation** | Triplet $\to$ Amino Acid | `[CODON_TABLE.get(seq[i:i+3], 'X') for i in range(0, len, 3)]` |
| **Wallace $T_m$** | Short oligos ($< 14$ nt) | `2 * (A + T) + 4 * (G + C)` |
| **Marmur $T_m$** | Longer oligos ($\ge 14$ nt) | `64.9 + 41.0 * (gc - 16.4) / n` |
| **Hamming Dist** | Point mismatch count | `sum(1 for a, b in zip(s1, s2) if a != b)` |
| **K-mers** | Substrings of length $k$ | `[seq[i:i+k] for i in range(len(seq) - k + 1)]` |
| **Jaccard Sim** | $\frac{|K_A \cap K_B|}{|K_A \cup K_B|}$ | `len(k1 & k2) / len(k1 \| k2)` |


## 7. Further Reading & References

1. **Lobry, J. R. (1996)**. Asymmetric substitution barriers in the two DNA strands of bacteria. *Molecular Biology and Evolution*, 13(5), 660-665.
2. **Marmur, J., & Doty, P. (1962)**. Determination of the base composition of deoxyribonucleic acid from its thermal denaturation temperature. *Journal of Molecular Biology*, 5(1), 109-118.
3. **Wallace, R. B., et al. (1979)**. Hybridization of synthetic oligodeoxyribonucleotides to phi chi 174 DNA. *Nucleic Acids Research*, 6(11), 3543-3557.
4. **Ondov, B. D., et al. (2016)**. Mash: fast genome and metagenome distance estimation using MinHash. *Genome Biology*, 17(1), 132.
